# LLM Judgement

Other language models rate **exactly the items the humans rated**, with **exactly the same four things and the same questions** (profile, recipe, checked facts, explanation; rubric from `rubric.json`). Their answers are saved in the same format as the human ratings, so notebook 10 can compare them directly.

**Works with any human file**: the pilot file, the main file, one annotator or many. The judges only see the `case_id`s that appear in the human file(s) you list in `HUMAN_FILES`. Nothing else is judged, so you never pay for items nobody rated.

**Inputs:** `annotations/annotations_<mode>_raw.csv` (Output of notebook 8), `case_bank/case_bank.jsonl` and `rubric.json` (Output of notebook 6).

**Output:** `processed/judgments/judgments_<judge>__<setup>.csv` (one file per judge x setup), a cache so a crash or restart never repeats paid/slow calls, and `judge_run_report.json`.

### What is a "setup"?
RQ4 asks *which way of judging catches which error*. A setup is a way of asking the same questions:
| setup | what the judge gets | why |
|---|---|---|
| `rubric_facts` | profile + recipe + **checked facts** + explanation, then the rubric | the same information a human annotator had (the fair comparison) |
| `rubric_nofacts` | the same but **without** the checked facts | shows how much the fact list helps |
| `checklist` | like `rubric_facts`, but the judge must first list each claim and check it, and check every stated requirement, *then* fill in the rubric | tests whether step-by-step checking catches more errors |

### Rules that keep the comparison fair
- Judges must be a **different model family from the explanation writer** (llama3.1). The notebook warns if not.
- The judges never see the technique, the answer key or the human ratings.
- Pairwise items are asked **twice, in both orders** (A/B swapped), to measure position bias.
- `provider: "mock"` fills in random answers so you can test the whole pipeline without any model. Do not report mock results.

In [1]:
import sys, re, json, time, random, hashlib, os
from pathlib import Path
from collections import Counter

import numpy as np, pandas as pd, requests


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Open the notebook from inside the FoodJudgeBench folder.")


ROOT = find_project_root(); sys.path.insert(0, str(ROOT / "src"))
from config import PROC_DIR

BANK = PROC_DIR / "case_bank"
JUDG = PROC_DIR / "judgments"; JUDG.mkdir(exist_ok=True)
print("project root:", ROOT)

project root: C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench


#### 1. Settings (the only cell you normally edit)

**Which human ratings define the items to judge?** List one or more files. Examples:
- pilot only: `["annotations_pilot_raw.csv"]`
- main round (one or more annotators): `["annotations_main_raw.csv"]`
- both: `["annotations_pilot_raw.csv", "annotations_main_raw.csv"]`

**Which judges?** Free local models via Ollama (`ollama pull <model>` first). Choose models that are **not llama**. For a hosted free API with an OpenAI-compatible endpoint (Groq, OpenRouter, Together, HF router, ...) use `provider: "openai_compat"`, a `base_url` and the name of an environment variable holding your key (never paste keys into the notebook).

In [2]:
import dotenv

dotenv.load_dotenv()

True

In [3]:
HUMAN_FILES = ["annotations_pilot_raw.csv"]       # files inside processed/annotations/ [annotations_pilot_raw.csv or annotations_main_raw.csv]
SETUPS = ["rubric_facts", "rubric_nofacts", "checklist"]
REPEATS = 1                  # run each item this many times (use 3 to measure judge stability; costs 3x)
LIMIT = None                 # e.g. 5 = only the first 5 items (quick test)
GENERATOR_FAMILIES = ["llama"]          # judges must not be from these families (the explanation writer)

JUDGES = [
    # name (used in file names), provider, model, base_url, key_env
    {"name": "qwen14b",  "provider": "ollama", "model": "qwen2.5:14b", "base_url": "http://localhost:11434"},   # NOTE: also the technique-4 auditor
    {"name": "gemma9b",  "provider": "ollama", "model": "gemma2:9b",   "base_url": "http://localhost:11434"},
    {"name": "mistral7b", "provider": "ollama", "model": "mistral:7b", "base_url": "http://localhost:11434"},
    # {"name": "groq70b", "provider": "openai_compat", "model": "llama-3.3-70b-versatile", "base_url": "https://api.groq.com/openai/v1", "key_env": "GROQ_API_KEY"},
    # {"name": "mock", "provider": "mock", "model": "mock"},       # pipeline test only, no LLM
]
TEMPERATURE = 0.1            # judges should be as deterministic as possible; with REPEATS > 1 later repeats use 0.3
MAX_TOKENS = 900
NUM_CTX = 4096               # Ollama context window (prompts are about 1.5k-2.5k tokens)
MIN_SECONDS_BETWEEN_CALLS = 0.0   # raise (e.g. 2.5) for free APIs with rate limits
TIMEOUT = 300

#### 2. Load what the humans rated, the items, and the rubric

In [4]:
rubric_spec = json.load(open(BANK / "rubric.json", encoding="utf-8"))
RUBRIC, PAIR_REASONS, COLS = rubric_spec["rubric"], rubric_spec["pair_reasons"], rubric_spec["columns"]
RMAP = {r["id"]: r for r in RUBRIC}

human = pd.concat([pd.read_csv(PROC_DIR / "annotations" / f, dtype=str, keep_default_na=False) for f in HUMAN_FILES], ignore_index=True)
item_type = human.drop_duplicates("case_id").set_index("case_id")["item_type"].to_dict()
ids = list(dict.fromkeys(human["case_id"]))
bank = {}
for l in open(BANK / "case_bank.jsonl", encoding="utf-8"):
    if l.strip():
        d = json.loads(l); bank[d["case_id"]] = d
missing = [i for i in ids if i not in bank]
assert not missing, f"{len(missing)} human-rated items are not in case_bank.jsonl (was notebook 6 re-run with another seed?): {missing[:5]}"
if LIMIT: ids = ids[:LIMIT]
print(f"{len(ids)} items to judge  ({Counter(item_type[i] for i in ids)});  human raters per item: {human.groupby('case_id')['annotator_id'].nunique().value_counts().to_dict()}")

10 items to judge  (Counter({'case': 8, 'diagnostic': 2}));  human raters per item: {3: 10}


#### 3. Prompts (the same text for every judge; only the setup changes)

In [5]:
SYSTEM = ("You are a careful, impartial evaluator of explanations written for food recommendations. "
          "You judge how correct, honest and suitable an explanation is FOR THE STATED PROFILE - never whether you like the dish. "
          "Trust the checked facts. Answer with a single JSON object and nothing else.")


def rubric_block():
    lines = []
    for r in RUBRIC:
        t = r["type"]
        allowed = {"yn": '"yes" or "no"', "yn_na": '"yes", "no" or "na"', "scale": "an integer 1-5", "scale_na": 'an integer 1-5 or "na"'}[t]
        na = f' Use "na" only when: {r["na"]}.' if r.get("na") else ""
        lines.append(f'- "{r["id"]}" ({allowed}): {r["q"]}{na}')
    return "\n".join(lines)


def card(item, facts=True):
    s = "PROFILE (what is known about the person)\n" + "\n".join("- " + x for x in item["profile_card"])
    s += f'\n\nRECIPE: {item["recipe"]["title"]}\nIngredients:\n' + "\n".join("- " + x for x in item["recipe"]["ingredients"])
    if facts:
        s += "\n\nCHECKED FACTS ABOUT THE RECIPE (trust these)\n" + "\n".join("- " + x for x in item["evidence"])
    return s


SCALE_NOTE = "Scales: 1 = very poor, 5 = excellent. Violation questions are about problems: answer \"yes\" only if the problem really is present."
KEYS_EXAMPLE = "{" + ", ".join(f'"{r["id"]}": ...' for r in RUBRIC) + "}"


def build_case_prompt(item, setup):
    facts = setup != "rubric_nofacts"
    body = card(item, facts) + f'\n\nEXPLANATION TO EVALUATE\n"""{item["explanation"]}"""\n\nQUESTIONS\n{rubric_block()}\n\n{SCALE_NOTE}\n'
    if setup == "checklist":
        body += ('\nWork step by step. First fill "claims": list every factual claim in the explanation about the recipe or the person, with a verdict '
                 '("supported" = stated by the profile/recipe/facts, "unsupported" = not stated anywhere, "contradicted" = conflicts with them) and a short reason. '
                 'Then fill "requirements": for each requirement the profile STATES (diet, allergy, time limit), say whether the recipe and the explanation respect it. '
                 'Only then give "ratings", derived from your checks.\n'
                 'Return JSON: {"claims": [{"claim": "...", "verdict": "...", "why": "..."}], "requirements": [{"requirement": "...", "recipe_ok": true, "explanation_ok": true}], "ratings": ' + KEYS_EXAMPLE + "}")
    else:
        body += "\nReturn JSON with exactly these keys: " + KEYS_EXAMPLE
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": body}]


def build_pair_prompt(item, setup, swap):
    a, b = (item["explanation_B"], item["explanation_A"]) if swap else (item["explanation_A"], item["explanation_B"])
    facts = setup != "rubric_nofacts"
    body = (card(item, facts) + f'\n\nEXPLANATION A\n"""{a}"""\n\nEXPLANATION B\n"""{b}"""\n\n'
            "Which explanation is better FOR THIS PERSON? Judge accuracy, honesty about downsides and fit with the profile - not which dish you prefer.\n"
            f'Return JSON: {{"pair_choice": "A" or "B" or "tie", "pair_reason": one of {json.dumps(PAIR_REASONS)}}}')
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": body}]


# show one prompt so you can read exactly what a judge sees
_demo = next(i for i in ids if item_type[i] != "pair")
print(build_case_prompt(bank[_demo], "checklist")[1]["content"][:3500])

PROFILE (what is known about the person)
- Profile specified explicitly for this benchmark (synthetic persona).
- States a liking for: soup.
- Dietary constraints: none.
- Health goal: unknown.
- Time budget: about 45 minutes or less.
- Unknown: dietary constraints, allergies, health goal.

RECIPE: Better Than Granny's and Easy Chicken and Dumplings
Ingredients:
- 2 cups rotisserie-cooked chicken (cooked, pulled off bones)
- 4 (14 ounce) cans chicken broth
- 2 (10 1/2 ounce) cans cream of celery soup
- 0.25 teaspoon pepper
- 0.5 teaspoon poultry seasoning
- 1 (10 count) can buttermilk biscuits

CHECKED FACTS ABOUT THE RECIPE (trust these)
- It takes about 35 minutes in total.
- Per serving: 827 kcal, 40 g protein, 40 g fat (13 g saturated), 75 g carbohydrate (14 g sugar, 3 g fibre), 3709 mg sodium.
- Compared with other recipes, this recipe is very high in calories, very high in sodium, high in sugar, high in saturated fat, very high in protein, moderate in fiber.
- Nutrition-score lev

#### 4. Model client (Ollama, OpenAI-compatible APIs, mock) with an on-disk cache

In [6]:
_last_call = [0.0]


def cache_path(j): return JUDG / f"cache_{j['name']}.jsonl"


def load_cache(j):
    c = {}
    p = cache_path(j)
    if p.exists():
        for l in open(p, encoding="utf-8"):
            if l.strip():
                try:
                    d = json.loads(l); c[d["k"]] = d["v"]
                except Exception:
                    pass
    return c


def cache_key(j, messages, temperature, repeat):
    options = {
        "temperature": temperature,
        "num_predict": MAX_TOKENS,
        "num_ctx": NUM_CTX,
        "repeat_penalty": 1.15,
        "repeat_last_n": 64,
    }
    return hashlib.sha256(json.dumps([j["provider"], j["model"], messages, temperature, options, repeat], sort_keys=True).encode()).hexdigest()


def mock_answer(messages):
    r = random.Random(hashlib.md5(json.dumps(messages).encode()).hexdigest())
    u = messages[-1]["content"]
    if "pair_choice" in u:
        return json.dumps({"pair_choice": r.choice(["A", "B", "tie"]), "pair_reason": r.choice(PAIR_REASONS)})
    out = {}
    for q in RUBRIC:
        t = q["type"]
        out[q["id"]] = r.choice(["yes", "no", "no"] + (["na"] if t == "yn_na" else [])) if t.startswith("yn") else r.choice(["1", "2", "3", "4", "5"] + (["na"] if t == "scale_na" else []))
    return json.dumps({"ratings": out}) if "claims" in u else json.dumps(out)


def call_model(j, messages, temperature):
    wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call[0])
    if wait > 0: time.sleep(wait)
    _last_call[0] = time.time()
    if j["provider"] == "mock":
        return mock_answer(messages)
    if j["provider"] == "ollama":
        body = {"model": j["model"], "messages": messages, "stream": False, "format": "json",
                "options": {"temperature": temperature, "num_predict": MAX_TOKENS, "num_ctx": NUM_CTX, "repeat_penalty": 1.15, "repeat_last_n": 64}}
        
        r = requests.post(
            j["base_url"].rstrip("/") + "/api/chat",
            json=body,
            timeout=TIMEOUT
        )

        if not r.ok:
            raise RuntimeError(
                f"Ollama request failed for judge={j['name']}, "
                f"model={j['model']}, HTTP {r.status_code}.\n"
                f"Response body: {r.text[:2000]}"
            )

        data = r.json()

        if "message" not in data or "content" not in data["message"]:
            raise RuntimeError(
                f"Unexpected Ollama response for {j['model']}: {data}"
            )

        return data["message"]["content"]
    
    if j["provider"] == "openai_compat":
        key = os.environ.get(j.get("key_env", ""), "")
        if not key: raise RuntimeError(f"environment variable {j.get('key_env')} is empty (put the API key there, not in the notebook)")
        body = {"model": j["model"], "messages": messages, "temperature": temperature, "max_tokens": MAX_TOKENS, "response_format": {"type": "json_object"}}
        for attempt in range(6):
            r = requests.post(j["base_url"].rstrip("/") + "/chat/completions", json=body, headers={"Authorization": f"Bearer {key}"}, timeout=TIMEOUT)
            if r.status_code == 400 and "response_format" in body:
                body.pop("response_format"); continue
            if r.status_code in (429, 500, 502, 503):
                time.sleep(min(60, 4 * 2 ** attempt)); continue
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        raise RuntimeError("API kept failing (rate limit?)")
    raise ValueError("unknown provider " + j["provider"])


def check_judge(j):
    fam = j["model"].lower()
    if any(g in fam for g in GENERATOR_FAMILIES):
        print(f"  WARNING {j['name']}: same family as the explanation writer ({GENERATOR_FAMILIES}); judges should come from other families")
    if j["provider"] == "ollama":
        try:
            tags = [m["name"] for m in requests.get(j["base_url"].rstrip("/") + "/api/tags", timeout=10).json().get("models", [])]
        except Exception as e:
            print(f"  {j['name']}: cannot reach Ollama at {j['base_url']} ({e}) -> skipped"); return False
        if not any(t == j["model"] or t.split(":")[0] == j["model"] and ":" not in j["model"] for t in tags):
            print(f"  {j['name']}: model not installed. Run:  ollama pull {j['model']}   -> skipped"); return False
    if j["provider"] == "mock":
        print("  MOCK judge: random answers, for testing the pipeline only")
    return True

#### 5. Parse and validate the model's answer

In [7]:
def extract_json(text):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    start = text.find("{")
    if start < 0: raise ValueError("no JSON object in the answer")
    depth, in_str, esc = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
        elif ch == '"': in_str = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return json.loads(text[start:i + 1])
    raise ValueError("unterminated JSON")


def norm(v):
    s = str(v).strip().lower().replace("not applicable", "na").replace("n/a", "na")
    if s in ("true",): s = "yes"
    if s in ("false",): s = "no"
    if re.fullmatch(r"\d(\.0+)?", s): s = str(int(float(s)))
    return s


def validate_ratings(d):
    d = d.get("ratings", d) if isinstance(d.get("ratings", d), dict) else d
    out, errs = {}, []
    for q in RUBRIC:
        v = norm(d.get(q["id"], ""))
        ok = {"yn": {"yes", "no"}, "yn_na": {"yes", "no", "na"}, "scale": set("12345"), "scale_na": set("12345") | {"na"}}[q["type"]]
        if v in ok: out[q["id"]] = v
        else: errs.append(f'{q["id"]}={d.get(q["id"], "MISSING")!r}')
    return out, errs


def validate_pair(d):
    c = str(d.get("pair_choice", "")).strip()
    c = {"a": "A", "b": "B", "tie": "tie", "equal": "tie"}.get(c.lower(), c)
    r = str(d.get("pair_reason", "")).strip().lower()
    r = next((x for x in PAIR_REASONS if x.lower() == r), "other")
    return ({"pair_choice": c, "pair_reason": r}, []) if c in ("A", "B", "tie") else ({}, [f"pair_choice={d.get('pair_choice')!r}"])

#### 6. Run the judges (resumable: re-run this cell after any interruption)

In [9]:
import ollama
ollama.pull("qwen2.5:14b")
ollama.pull("gemma2:9b")
ollama.pull("mistral:7b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [10]:
def judge_one(j, cache, messages, repeat, is_pair):
    temp = TEMPERATURE if repeat == 0 else 0.3
    last = ""
    for attempt in range(3):
        msgs = messages if attempt == 0 else messages + [{"role": "assistant", "content": last}, {"role": "user", "content": f"Your answer was not valid ({err}). Return ONLY the corrected JSON object."}]
        k = cache_key(j, msgs, temp, repeat)
        if k in cache:
            raw_text = cache[k]
        else:
            raw_text = call_model(j, msgs, temp)
            cache[k] = raw_text
            with open(cache_path(j), "a", encoding="utf-8") as f:
                f.write(json.dumps({"k": k, "v": raw_text}, ensure_ascii=False) + "\n")
        last = raw_text
        try:
            d = extract_json(raw_text)
            vals, errs = validate_pair(d) if is_pair else validate_ratings(d)
            if not errs: return vals, raw_text, ""
            err = "; ".join(errs[:4])
        except Exception as e:
            err = str(e)[:120]
    return {}, last, err


def run_all():
    report = {"items": len(ids), "setups": SETUPS, "repeats": REPEATS, "judges": {}}
    for j in JUDGES:
        print(f"\n== judge {j['name']} ({j['provider']}:{j['model']})")

        if not check_judge(j):
            report["judges"][j["name"]] = "skipped"; continue
        cache = load_cache(j)
        for setup in SETUPS:
            print(f"\n   starting setup={setup} for judge={j['name']} ({j['model']})")
            out_path = JUDG / f"judgments_{j['name']}__{setup}.csv"
            done = set()
            if out_path.exists():
                old = pd.read_csv(out_path, dtype=str, keep_default_na=False)
                done = set(zip(old["case_id"], old["repeat"], old["pair_order"]))
            rows, n_fail, t0 = [], 0, time.time()
            for n, cid in enumerate(ids):
                it = bank[cid]; is_pair = it["item_type"] == "pair"
                for rep in range(REPEATS):
                    for order in (["AB", "BA"] if is_pair else [""]):
                        if (cid, str(rep), order) in done: continue
                        print(
                            f"   judging case {cid} "
                            f"({it['item_type']}), setup={setup}, "
                            f"judge={j['name']}",
                            flush=True
                        )
                        msgs = build_pair_prompt(it, setup, order == "BA") if is_pair else build_case_prompt(it, setup)
                        vals, raw_text, err = judge_one(j, cache, msgs, rep, is_pair)
                        row = {c: "" for c in COLS}
                        row.update({"annotator_id": f"{j['name']}|{setup}", "mode": "judge", "case_id": cid, "item_type": it["item_type"], "saved_at": pd.Timestamp.now(tz="UTC").isoformat()})
                        row.update(vals)
                        if is_pair and vals.get("pair_choice") in ("A", "B") and order == "BA":
                            row["pair_choice"] = {"A": "B", "B": "A"}[vals["pair_choice"]]      # map back to the original A/B labels
                        row.update({"judge": j["name"], "model": j["model"], "setup": setup, "repeat": rep, "pair_order": order, "parse_error": err})
                        n_fail += bool(err); rows.append(row)
                if (n + 1) % 10 == 0:
                    print(f"   {setup}: {n+1}/{len(ids)} items  ({time.time()-t0:.0f}s)")
            if rows:
                new = pd.DataFrame(rows)
                new.to_csv(out_path, mode="a", header=not out_path.exists(), index=False)
            report["judges"].setdefault(j["name"], {})[setup] = {"new_rows": len(rows), "unparseable": n_fail}
            print(f"   {setup}: +{len(rows)} rows, {n_fail} could not be parsed after 3 tries (they stay empty and are ignored in the next step)")
    return report

In [11]:
report = run_all()
(JUDG / "judge_run_report.json").write_text(json.dumps(report, indent=1))
print("\nfiles:", sorted(p.name for p in JUDG.glob("judgments_*.csv")))


== judge qwen14b (ollama:qwen2.5:14b)

   starting setup=rubric_facts for judge=qwen14b (qwen2.5:14b)
   judging case C-A876B (case), setup=rubric_facts, judge=qwen14b
   judging case C-C9CF9 (case), setup=rubric_facts, judge=qwen14b
   judging case C-16004 (case), setup=rubric_facts, judge=qwen14b
   judging case C-1102C (case), setup=rubric_facts, judge=qwen14b
   judging case C-A7134 (case), setup=rubric_facts, judge=qwen14b
   judging case C-37157 (case), setup=rubric_facts, judge=qwen14b
   judging case C-A65BE (case), setup=rubric_facts, judge=qwen14b
   judging case C-1B7F4 (case), setup=rubric_facts, judge=qwen14b
   judging case C-A762B (diagnostic), setup=rubric_facts, judge=qwen14b
   judging case C-CF2B8 (diagnostic), setup=rubric_facts, judge=qwen14b
   rubric_facts: 10/10 items  (182s)
   rubric_facts: +10 rows, 0 could not be parsed after 3 tries (they stay empty and are ignored in the next step)

   starting setup=rubric_nofacts for judge=qwen14b (qwen2.5:14b)
   judgi

#### 7. Quick sanity check of the judges' answers

In [12]:
fs = sorted(JUDG.glob("judgments_*.csv"))
if fs:
    jd = pd.concat([pd.read_csv(f, dtype=str, keep_default_na=False) for f in fs], ignore_index=True)
    jd = jd[jd["case_id"].isin(ids)]
    print("rows per judge|setup:\n", jd.groupby("annotator_id").size().to_string())
    print("\nshare of unparseable answers per judge|setup (should be near 0):")
    print((jd["parse_error"] != "").groupby(jd["annotator_id"]).mean().round(3).to_string())
    case_rows = jd[jd["item_type"] != "pair"]
    print("\n'yes' rate on the violation questions per judge|setup (humans: see notebook 8):")
    print(case_rows.groupby("annotator_id")[[r["id"] for r in RUBRIC if r["type"].startswith("yn")]].agg(lambda s: round((s == "yes").mean(), 2)).to_string())
    print("\nNext: run 10_Analysis_and_Release.ipynb")

rows per judge|setup:
 annotator_id
gemma9b|checklist           10
gemma9b|rubric_facts        10
gemma9b|rubric_nofacts      10
mistral7b|checklist         10
mistral7b|rubric_facts      10
mistral7b|rubric_nofacts    10
qwen14b|checklist           10
qwen14b|rubric_facts        10
qwen14b|rubric_nofacts      10

share of unparseable answers per judge|setup (should be near 0):
annotator_id
gemma9b|checklist           0.0
gemma9b|rubric_facts        0.0
gemma9b|rubric_nofacts      0.0
mistral7b|checklist         0.7
mistral7b|rubric_facts      0.3
mistral7b|rubric_nofacts    0.3
qwen14b|checklist           0.0
qwen14b|rubric_facts        0.0
qwen14b|rubric_nofacts      0.0

'yes' rate on the violation questions per judge|setup (humans: see notebook 8):
                          ignored_constraint  factual_error  invented_personalization  unsupported_claim  overstated_certainty
annotator_id                                                                                                  